# 08 Finales Analyse-Dataset joinen

Dieses Notebook implementiert BD-17 und BD-18. Es verbindet das Team-Match-Basisdataset mit den Wetterdaten aus dem Kafka-Snapshot und den bereits berechneten Elo-Features. Anschliessend werden Data-Quality-Regeln fuer das finale Analyse-Dataset geprueft.

Outputs:

- `data/gold/team_match_analysis_dataset.parquet/`
- `data/gold/team_match_analysis_dataset.csv`
- `outputs/tables/bd17_missing_values.csv`
- `outputs/tables/bd17_join_quality_summary.csv`
- `outputs/tables/data_quality_summary.csv`


## Methodischer Ansatz

Der Join ist als DataFrame-orientierter Transformationsschritt aufgebaut: Datenquellen werden als tabellarische DataFrames gelesen, Join-Keys werden vorab auf Eindeutigkeit geprueft, und der finale Datensatz entsteht mit deklarativen `select`-, `join`- und Aggregationsoperationen.

Fachlich bleibt die Analyseebene `team_match`: Ein Spiel erzeugt zwei Zeilen, eine pro Team. Wetterdaten liegen auf Match-Ebene und werden deshalb ueber `match_id` an beide Team-Zeilen gejoint. Elo-Features liegen bereits auf Team-Match-Ebene und werden ueber `match_id` und `team_id` verbunden.

In [ ]:
import os
import shutil

import pandas as pd

from project_paths import ensure_pyspark_path, project_root as get_project_root
from pipeline_config import DEFAULT_SPARK_MASTER

ensure_pyspark_path()

from pyspark.sql import SparkSession, functions as F

SPARK_MASTER = os.getenv('SPARK_MASTER', DEFAULT_SPARK_MASTER)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
gold_path = base_path / 'data' / 'gold'
tables_path = base_path / 'outputs' / 'tables'

team_match_base_path = silver_path / 'team_match_base.parquet'
weather_path = silver_path / 'weather_from_kafka.parquet'
elo_features_path = silver_path / 'team_match_elo_features.parquet'
elo_history_path = bronze_path / 'elo_ratings.parquet'

parquet_output_path = gold_path / 'team_match_analysis_dataset.parquet'
csv_output_path = gold_path / 'team_match_analysis_dataset.csv'
missing_values_output_path = tables_path / 'bd17_missing_values.csv'
join_quality_output_path = tables_path / 'bd17_join_quality_summary.csv'
data_quality_summary_output_path = tables_path / 'data_quality_summary.csv'

for path in [gold_path, tables_path]:
    path.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('football-weather-bd17-final-dataset')
    .master(SPARK_MASTER)
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

{
    'team_match_base_path': str(team_match_base_path),
    'weather_path': str(weather_path),
    'elo_features_path': str(elo_features_path),
    'elo_history_path': str(elo_history_path),
    'parquet_output_path': str(parquet_output_path),
    'csv_output_path': str(csv_output_path),
    'spark_master': SPARK_MASTER,
}

## Inputs laden

`team_match_base` enthaelt die StatsBomb-Teammetriken und Match-Metadaten. `weather_from_kafka` ist ein deduplizierter Match-Snapshot aus Kafka. `team_match_elo_features` enthaelt die in BD-08 berechneten Team- und Gegner-Elo-Werte. Der Elo-Input wurde in BD-08 mit Pandas geschrieben; Datumsfelder werden deshalb vor der Spark-Verarbeitung normalisiert, damit Spark nicht ueber Nanosekunden-Timestamps aus dem Parquet-Schema stolpert.

In [ ]:
required_inputs = [team_match_base_path, weather_path, elo_features_path, elo_history_path]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
assert not missing_inputs, f'Missing required input files: {missing_inputs}'

team_match_base = spark.read.parquet(str(team_match_base_path))
weather = spark.read.parquet(str(weather_path))

elo_features_pd = pd.read_parquet(elo_features_path)
elo_history_pd = pd.read_parquet(elo_history_path)

for date_column in ['match_date', 'team_elo_match_date', 'opponent_elo_match_date']:
    if date_column in elo_features_pd.columns:
        elo_features_pd[date_column] = pd.to_datetime(elo_features_pd[date_column]).dt.date.astype('string')

elo_features_pd['elo_missing_reason'] = elo_features_pd['elo_missing_reason'].fillna('none').astype('string')
for string_column in elo_features_pd.select_dtypes(include='string').columns:
    elo_features_pd[string_column] = elo_features_pd[string_column].astype(object)

elo_features = spark.createDataFrame(elo_features_pd)

input_counts = {
    'team_match_base_rows': team_match_base.count(),
    'team_match_base_matches': team_match_base.select('match_id').distinct().count(),
    'weather_rows': weather.count(),
    'weather_matches': weather.select('match_id').distinct().count(),
    'elo_feature_rows': elo_features.count(),
    'elo_feature_team_matches': elo_features.select('match_id', 'team_id').distinct().count(),
    'elo_history_rows': len(elo_history_pd),
}

team_match_base.select('match_id', 'match_date', 'stadium_name', 'team_id', 'team_name', 'opponent_team_name').orderBy('match_id', F.desc('is_home')).show(6, truncate=False)
input_counts

## Join-Keys pruefen

Vor dem eigentlichen Join wird geprueft, ob die erwarteten Schluessel eindeutig sind. Wetterdaten muessen eine Zeile pro `match_id` haben. Team- und Elo-Daten muessen eine Zeile pro `(match_id, team_id)` haben.

In [ ]:
def duplicate_key_count(df, key_columns):
    return (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col('count') > 1)
        .count()
    )


join_key_checks = {
    'duplicate_team_match_base_keys': duplicate_key_count(team_match_base, ['match_id', 'team_id']),
    'duplicate_weather_match_keys': duplicate_key_count(weather, ['match_id']),
    'duplicate_elo_team_match_keys': duplicate_key_count(elo_features, ['match_id', 'team_id']),
    'matches_without_two_team_rows': (
        team_match_base.groupBy('match_id')
        .count()
        .filter(F.col('count') != 2)
        .count()
    ),
}

assert join_key_checks['duplicate_team_match_base_keys'] == 0
assert join_key_checks['duplicate_weather_match_keys'] == 0
assert join_key_checks['duplicate_elo_team_match_keys'] == 0
assert join_key_checks['matches_without_two_team_rows'] == 0

join_key_checks

## Wetter- und Elo-Spalten vorbereiten

Wetterdaten behalten eigene Prefix-Spalten fuer Metadaten, damit sie nicht mit den kanonischen Match-Metadaten aus StatsBomb kollidieren. Bei Elo werden nur die Teamstaerke-Features uebernommen, weil Match- und Teammetadaten bereits in `team_match_base` vorhanden sind.

In [ ]:
weather_features = weather.select(
    F.col('match_id').cast('long').alias('match_id'),
    F.col('match_date').alias('weather_match_date'),
    F.col('kick_off').alias('weather_kick_off'),
    'kickoff_timestamp',
    'weather_time',
    'hours_from_kickoff',
    'weather_lookup_key',
    F.col('stadium_id').cast('long').alias('weather_stadium_id'),
    F.col('stadium_name').alias('weather_stadium_name'),
    F.col('city_name').alias('weather_city_name'),
    F.col('country_name').alias('weather_country_name'),
    'latitude',
    'longitude',
    'openmeteo_timezone',
    'temperature_c',
    'feels_like_c',
    'precipitation_mm',
    'rain_mm',
    'rain_flag',
    'temperature_group',
    'weather_missing_flag',
    'kafka_topic',
    'kafka_partition',
    'kafka_offset',
    'kafka_timestamp',
)

elo_join_features = elo_features.select(
    F.col('match_id').cast('long').alias('match_id'),
    F.col('team_id').cast('long').alias('team_id'),
    F.col('team_name').alias('elo_team_match_team_name'),
    F.col('opponent_team_name').alias('elo_team_match_opponent_name'),
    'team_elo',
    'team_elo_match_date',
    'team_elo_team_name',
    'team_elo_team_code',
    'team_elo_source_page',
    'opponent_elo',
    'opponent_elo_match_date',
    'opponent_elo_team_name',
    'opponent_elo_team_code',
    'opponent_elo_source_page',
    'elo_diff',
    'elo_group',
    'elo_missing_reason',
)

weather_features.orderBy('match_id').show(5, truncate=False)
elo_join_features.orderBy('match_id', F.desc('team_id')).show(5, truncate=False)

## Finalen Datensatz bauen

Die Join-Reihenfolge ist bewusst einfach: zuerst wird Wetter auf Match-Ebene ergaenzt, danach Elo auf Team-Match-Ebene. Dadurch bleibt die Zeilenanzahl des Basisdatasets erhalten.

In [ ]:
final_dataset = (
    team_match_base.alias('base')
    .join(weather_features.alias('weather'), on='match_id', how='left')
    .join(elo_join_features.alias('elo'), on=['match_id', 'team_id'], how='left')
    .withColumn('weather_join_missing', F.col('temperature_c').isNull())
    .withColumn('elo_join_missing', F.col('team_elo').isNull() | F.col('opponent_elo').isNull())
    .withColumn(
        'weather_date_matches_statsbomb',
        F.to_date('match_date') == F.to_date('weather_match_date'),
    )
    .withColumn(
        'weather_stadium_matches_statsbomb',
        F.col('stadium_id') == F.col('weather_stadium_id'),
    )
    .withColumn(
        'elo_team_name_matches_statsbomb',
        F.col('team_name') == F.col('elo_team_match_team_name'),
    )
    .withColumn(
        'elo_opponent_name_matches_statsbomb',
        F.col('opponent_team_name') == F.col('elo_team_match_opponent_name'),
    )
    .orderBy('match_id', F.desc('is_home'))
)

join_counts = {
    'base_rows': team_match_base.count(),
    'final_rows': final_dataset.count(),
    'base_team_match_keys': team_match_base.select('match_id', 'team_id').distinct().count(),
    'final_team_match_keys': final_dataset.select('match_id', 'team_id').distinct().count(),
    'weather_join_missing_rows': final_dataset.filter(F.col('weather_join_missing')).count(),
    'elo_join_missing_rows': final_dataset.filter(F.col('elo_join_missing')).count(),
    'weather_date_mismatch_rows': final_dataset.filter(~F.col('weather_date_matches_statsbomb')).count(),
    'weather_stadium_mismatch_rows': final_dataset.filter(~F.col('weather_stadium_matches_statsbomb')).count(),
    'elo_team_name_mismatch_rows': final_dataset.filter(~F.col('elo_team_name_matches_statsbomb')).count(),
    'elo_opponent_name_mismatch_rows': final_dataset.filter(~F.col('elo_opponent_name_matches_statsbomb')).count(),
}

assert join_counts['final_rows'] == join_counts['base_rows']
assert join_counts['final_team_match_keys'] == join_counts['base_team_match_keys']
assert join_counts['weather_join_missing_rows'] == 0
assert join_counts['elo_join_missing_rows'] == 0
assert join_counts['weather_date_mismatch_rows'] == 0
assert join_counts['weather_stadium_mismatch_rows'] == 0
assert join_counts['elo_team_name_mismatch_rows'] == 0
assert join_counts['elo_opponent_name_mismatch_rows'] == 0

final_dataset.select(
    'match_id', 'short_name', 'match_date', 'stadium_name', 'team_name', 'opponent_team_name',
    'xg', 'temperature_c', 'rain_flag', 'team_elo', 'opponent_elo', 'elo_diff', 'elo_group'
).show(10, truncate=False)
join_counts

## Fehlende Werte dokumentieren

Die Missing-Value-Tabelle dokumentiert alle Spalten mit fehlenden Werten. Das ist der direkte Anschluss fuer BD-18, wo aus dieser Sicht weitere Data-Quality-Regeln abgeleitet werden.

In [ ]:
row_count = final_dataset.count()
missing_exprs = [F.sum(F.col(column).isNull().cast('long')).alias(column) for column in final_dataset.columns]
missing_counts = final_dataset.select(missing_exprs).collect()[0].asDict()

missing_values = pd.DataFrame(
    [
        {
            'column': column,
            'missing_count': int(count),
            'missing_share': float(count) / row_count if row_count else 0.0,
        }
        for column, count in missing_counts.items()
    ]
).sort_values(['missing_count', 'column'], ascending=[False, True])

missing_values.to_csv(missing_values_output_path, index=False)
missing_values.head(20)

## BD-18: Data-Quality-Regeln pruefen

Die Regeln sind als DataFrame-Pruefungen formuliert: Jede Qualitaetsfrage wird als filter-, groupBy- oder Aggregationsausdruck beschrieben und als zaehlbarer Befund in einer Summary-Tabelle abgelegt. Kritische Befunde blockieren das finale Dataset; nicht-kritische Befunde werden begruendet.


In [ ]:
def count_where(df, condition):
    return df.filter(condition).count()


def missing_any_count(df, columns):
    condition = None
    for column in columns:
        column_missing = F.col(column).isNull()
        condition = column_missing if condition is None else condition | column_missing
    return count_where(df, condition)


data_quality_rules = [
    {
        'check': 'duplicate_team_match_rows',
        'severity': 'critical',
        'issue_count': duplicate_key_count(final_dataset, ['match_id', 'team_id']),
        'rule': 'Eine Zeile pro match_id/team_id.',
        'action': 'Keine Bereinigung noetig; Join-Keys sind eindeutig.',
    },
    {
        'check': 'missing_match_id',
        'severity': 'critical',
        'issue_count': count_where(final_dataset, F.col('match_id').isNull()),
        'rule': 'match_id muss fuer jede Team-Match-Zeile vorhanden sein.',
        'action': 'Keine Bereinigung noetig; alle Zeilen sind einem Match zugeordnet.',
    },
    {
        'check': 'missing_weather_values',
        'severity': 'critical',
        'issue_count': missing_any_count(
            final_dataset,
            ['temperature_c', 'feels_like_c', 'precipitation_mm', 'rain_mm', 'rain_flag', 'temperature_group'],
        ),
        'rule': 'Kern-Wetterwerte duerfen im finalen Dataset nicht fehlen.',
        'action': 'Keine Bereinigung noetig; Wetterdaten wurden vollstaendig gejoint.',
    },
    {
        'check': 'missing_elo_values',
        'severity': 'critical',
        'issue_count': missing_any_count(final_dataset, ['team_elo', 'opponent_elo', 'elo_diff', 'elo_group']),
        'rule': 'Team- und Gegner-Elo muessen fuer jede Team-Match-Zeile vorhanden sein.',
        'action': 'Keine Bereinigung noetig; Elo-Features sind vollstaendig.',
    },
    {
        'check': 'unrealistic_temperature_c',
        'severity': 'critical',
        'issue_count': count_where(final_dataset, (F.col('temperature_c') < -30) | (F.col('temperature_c') > 55)),
        'rule': 'Temperatur muss in einem plausiblen Fussball-Wetterbereich von -30 bis 55 Grad Celsius liegen.',
        'action': 'Keine Bereinigung noetig; alle Temperaturen liegen im plausiblen Bereich.',
    },
    {
        'check': 'negative_xg_values',
        'severity': 'critical',
        'issue_count': count_where(final_dataset, (F.col('xg') < 0) | (F.col('opponent_xg') < 0)),
        'rule': 'xG-Werte duerfen nicht negativ sein.',
        'action': 'Keine Bereinigung noetig; alle xG-Werte sind nicht-negativ.',
    },
    {
        'check': 'pass_completion_outside_0_1',
        'severity': 'critical',
        'issue_count': count_where(
            final_dataset,
            (F.col('pass_completion_rate') < 0)
            | (F.col('pass_completion_rate') > 1)
            | (F.col('opponent_pass_completion_rate') < 0)
            | (F.col('opponent_pass_completion_rate') > 1),
        ),
        'rule': 'Passquoten muessen zwischen 0 und 1 liegen.',
        'action': 'Keine Bereinigung noetig; alle Passquoten liegen im gueltigen Intervall.',
    },
    {
        'check': 'weather_join_missing_flag',
        'severity': 'critical',
        'issue_count': count_where(final_dataset, F.col('weather_join_missing') | F.col('weather_missing_flag')),
        'rule': 'Wetter-Join und vorgelagerter Wetter-Snapshot duerfen keine Missing-Flags tragen.',
        'action': 'Keine Bereinigung noetig; weder Join noch Wetter-Snapshot melden fehlende Wetterdaten.',
    },
    {
        'check': 'elo_join_missing_flag',
        'severity': 'critical',
        'issue_count': count_where(final_dataset, F.col('elo_join_missing') | (F.col('elo_missing_reason') != 'none')),
        'rule': 'Elo-Join darf keine fehlenden Teamstaerke-Werte enthalten.',
        'action': 'Keine Bereinigung noetig; Elo-Missing-Reason ist durchgehend none.',
    },
]

data_quality_summary = pd.DataFrame(data_quality_rules)
data_quality_summary['status'] = data_quality_summary['issue_count'].apply(lambda count: 'pass' if count == 0 else 'fail')
data_quality_summary = data_quality_summary[
    ['check', 'severity', 'status', 'issue_count', 'rule', 'action']
]

data_quality_summary.to_csv(data_quality_summary_output_path, index=False)

critical_quality_failures = data_quality_summary.query("severity == 'critical' and issue_count > 0")
assert critical_quality_failures.empty, critical_quality_failures.to_string(index=False)
assert data_quality_summary_output_path.exists()

data_quality_summary


## Speichern und Akzeptanzkriterien pruefen

Parquet ist das primaere Analyseformat in `data/gold/`. Zusaetzlich wird eine CSV-Datei geschrieben, damit die Daten ohne Spark schnell inspiziert werden koennen.

In [ ]:
if parquet_output_path.exists():
    if parquet_output_path.is_dir():
        shutil.rmtree(parquet_output_path)
    else:
        parquet_output_path.unlink()

if csv_output_path.exists():
    if csv_output_path.is_dir():
        shutil.rmtree(csv_output_path)
    else:
        csv_output_path.unlink()

final_dataset.write.mode('overwrite').parquet(str(parquet_output_path))
final_pd = final_dataset.toPandas()
final_pd.to_csv(csv_output_path, index=False)

written = spark.read.parquet(str(parquet_output_path))
written_rows = written.count()
written_team_match_keys = written.select('match_id', 'team_id').distinct().count()

required_feature_columns = [
    'xg',
    'shots',
    'passes',
    'temperature_c',
    'feels_like_c',
    'precipitation_mm',
    'rain_flag',
    'team_elo',
    'opponent_elo',
    'elo_diff',
    'elo_group',
]
missing_required_feature_columns = [column for column in required_feature_columns if column not in written.columns]

quality_summary = {
    **input_counts,
    **join_key_checks,
    **join_counts,
    'written_rows': written_rows,
    'written_team_match_keys': written_team_match_keys,
    'written_columns': len(written.columns),
    'missing_required_feature_columns': ','.join(missing_required_feature_columns),
    'parquet_output_path': 'data/gold/team_match_analysis_dataset.parquet',
    'csv_output_path': 'data/gold/team_match_analysis_dataset.csv',
    'missing_values_output_path': 'outputs/tables/bd17_missing_values.csv',
    'data_quality_summary_output_path': 'outputs/tables/data_quality_summary.csv',
    'data_quality_checks': len(data_quality_summary),
    'data_quality_failed_checks': int((data_quality_summary['issue_count'] > 0).sum()),
}

pd.DataFrame([quality_summary]).to_csv(join_quality_output_path, index=False)

assert parquet_output_path.exists()
assert csv_output_path.exists()
assert missing_values_output_path.exists()
assert join_quality_output_path.exists()
assert data_quality_summary_output_path.exists()
assert written_rows == row_count
assert written_team_match_keys == row_count
assert not missing_required_feature_columns

written.select(
    'match_id', 'short_name', 'team_name', 'opponent_team_name', 'xg',
    'temperature_c', 'temperature_group', 'team_elo', 'opponent_elo', 'elo_diff', 'elo_group'
).orderBy('match_id', F.desc('is_home')).show(12, truncate=False)
quality_summary

## Ergebnis

BD-17 ist erfuellt, wenn die Assertions erfolgreich sind und die Gold-Artefakte geschrieben wurden. BD-18 ist erfuellt, wenn `outputs/tables/data_quality_summary.csv` erzeugt wurde und alle kritischen Data-Quality-Regeln den Status `pass` haben. Das finale Dataset enthaelt StatsBomb-Spielstilmetriken, Wettervariablen aus Kafka und Elo-Teamstaerke auf Team-Match-Ebene.
